In [1]:
import sys
import numpy as np
import pybullet as p
import time
import logging
from typing import List, Tuple, Optional, Sequence, Collection, Dict, Any, cast
import random
import json
import ipdb

from predicators.structs import Action, Array, GroundAtom, Object, State, Type, ParameterizedOption
from predicators import utils
from predicators.settings import CFG
from gym.spaces import Box

#Import core environment methods, robot function etc.

from predicators.envs.pybullet_blocks import PyBulletBlocksEnv
from predicators.envs.pybullet_multitable_blocks import PyBulletMultiTableBlocksEnv
from predicators.envs.pybullet_env import PyBulletEnv, create_pybullet_block
from predicators.pybullet_helpers.robots import SingleArmPyBulletRobot
from predicators.pybullet_helpers.robots.mobile_single_arm import MobileSingleArmPyBulletRobot
from predicators.pybullet_helpers.geometry import Pose
from predicators.pybullet_helpers.joint import JointPositions, get_joint_infos, get_joint_positions
from predicators.pybullet_helpers.link import get_link_state, get_link_pose

#Import the functions that are to be tested:

from predicators.pybullet_helpers.motion_planning import run_motion_planning, run_base_motion_planning,\
                                                            run_coordinated_motion_planning
#The pick/place options to be tested are accessed via the env instance
from predicators.pybullet_helpers.controllers import execute_coordinated_path, create_move_end_effector_to_pose_option,\
                                                    create_change_fingers_option, create_move_base_option
#Configure logging for better debugging outputs:
#logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.basicConfig(
    level=logging.WARNING,                    
    format="%(asctime)s %(name)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

pybullet build time: Jan 29 2025 23:16:28


In [2]:
#Defining test configuration, and overriding some default ones:
CFG.pybullet_robot = "fetch_mobile"
CFG.use_gui = True
#Draws helpful debug lines in the workspace.
#NOT SURE WHETHER TO USE THIS. WILL DECIDE AFTER A COUPLE RUNS.
#CFG.pybullet_draw_debug = True
#Initializing with standard size of blocks.
CFG.blocks_block_size = 0.05
CFG.pybullet_birrt_num_iters = 50
CFG.pybullet_birrt_num_attempts = 10
CFG.pybullet_birrt_smooth_amt = 20
CFG.seed = random.randint(0,10000)
#CFG.seed = 12
#Num of PyBullet physics steps per high-level Action in visualize_action_sequence
# CFG.pybullet_sim_steps_per_action = 120

In [3]:
#Function to reset robot to a known pose
def reset_robot_fetch_mobile(robot: MobileSingleArmPyBulletRobot,
                             physics_client_id:int,
                             base_pose: Tuple[float, float, float] = (1.35, 0.75, 0.0), # (x,y,theta)
                             arm_joint_angle: Optional[List[float]]=None):
    """
    Resets the robot's base/ teleports it and arm to specified poses.

    """

    robot.move_base_to(base_pose, physics_client_id)
    if arm_joint_angle:
        #Set arm joints only
        robot.set_joints(arm_joint_angle)
    else:
        #robot.initial_joint_positions includes arm and finger joints
        robot.set_joints(robot.initial_joint_positions)
    #Step simulation a bit to allow PyBullet to settle the state.
    for _ in range(10):
        p.stepSimulation(physicsClientId=physics_client_id)


#Fn to create blocks in the env.
def create_test_block(env: PyBulletEnv,
                      pose: Tuple[float, float, float],
                      color: Tuple[float, float, float, float] = (0.8, 0.2, 0.2, 1.0),
                      name_suffix: str = "test") -> int:
    """
    Creates a single block at a specified pose for testing and returns its PyBullet ID.
    """

    #Use the fn defined in utils to create block
    block_id = create_pybullet_block(
        color,
        (CFG.blocks_block_size/2,)*3,
        env._obj_mass,
        env._obj_friction,
        env._default_orn,
        env._physics_client_id
    )
    #Place the block at desired pose.
    p.resetBasePositionAndOrientation(block_id, pose, env._default_orn, physicsClientId=env._physics_client_id)

    return block_id


#Get the list of all bodies except the robot.
#TODO: Need to add logic that saves object/body name
#      which can be used for better debugging with collision.
def get_all_non_robot_bodies(robot_id: int, physics_client_id:int) -> List[int]:
    """
    Gets all PyBullet body IDs in the simulation except for the robot itself.
    These are typically used as collision obstacles.
    """
    all_bodies = [p.getBodyUniqueId(i, physicsClientId=physics_client_id)
                    for i in range(p.getNumBodies(physicsClientId=physics_client_id))]


    return [b for b in all_bodies if b!=robot_id]

In [4]:
#Setup Env.
#Initialize the PyBulletBlocksEnv which sets up PyBullet,
#loads the robot, tables etc.

env = PyBulletBlocksEnv(use_gui=CFG.use_gui)

/home/cloaked04/anaconda3/envs/predicators/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [5]:
# Resets the environment to a specific task, getting an initial symbolic state.
# While initial_state_from_env is fetched, the option tests will create their own
# more specific symbolic states.
initial_state = env.reset("train", 0)

# The robot instance from the environment
robot = env._pybullet_robot
# The PyBullet physics client ID
physics_client_id = env._physics_client_id

if not isinstance(robot, MobileSingleArmPyBulletRobot):
    logging.error("This test script is designed for a MobileSingleArmPyBulletRobot.")

# dyn = p.getDynamicsInfo(robot.robot_id, -1, physicsClientId=env._physics_client_id)
# print(f"\nMass, inertialFrame…{dyn}")
# input()


logging.info(f"Using robot: {robot.get_name()}")


#Store the robot's default arm and finger joint positions.
home_arm_joints = robot.initial_joint_positions

robot_obj = initial_state.get_objects(env._robot_type)[0]
block_obj = initial_state.get_objects(env._block_type)[0]

#Store permament, fixed bodies
static_collision_bodies = get_all_non_robot_bodies(robot.robot_id, physics_client_id)

#print(f"Static_collision_bodies:{static_collision_bodies}")

#sys.exit(0)

#Define a rectangular workspace for base motion planning tests.
#(min_x, min_y, max_x, max_y)
workspace_bounds = (1.0, 0.2, 1.7, 1.3)

In [6]:
env._current_observation

PyBulletState(data={robby:robot: array([1.       , 0.3      , 0.5000001, 1.       ], dtype=float32), block0:block: array([1.3344929 , 0.899111  , 0.225     , 0.        , 0.00910209,
       0.5690874 , 0.68499064], dtype=float32), block1:block: array([1.3344929 , 0.899111  , 0.275     , 0.        , 0.25334013,
       0.25400946, 0.0106728 ], dtype=float32), block2:block: array([1.3344929 , 0.899111  , 0.325     , 0.        , 0.10069747,
       0.03853572, 0.62274086], dtype=float32), block3:block: array([1.3344929 , 0.899111  , 0.375     , 0.        , 0.6628766 ,
       0.05966876, 0.5627126 ], dtype=float32)}, simulator_state=[-0.5762533077569025, 0.21134056241183785, -1.2723320946144927, -1.5971479585602872, -1.7837994696985786, -1.8680940294583634, 0.9569871452140434, 0.03999999910593033, 0.03999999910593033], base_pose=(0.4, 0.3, 0.0))

In [7]:
initial_base_pose = (0.4, 1.6, -np.pi/2)
target_base_pose = (0.75, 0.7441, np.pi / 4) # Target base pose
logging.info(f"Testing base-only motion from {initial_base_pose} to {target_base_pose}...")

#Resetting robot's state:
robot.move_base_to(initial_base_pose, physics_client_id)
robot.set_joints(home_arm_joints)

#Step simulation a bit to allow PyBullet to settle the state.
for _ in range(20):
    p.stepSimulation(physicsClientId=physics_client_id)

In [8]:
static_collision_bodies

[0, 2, 3, 4, 5, 6, 7, 8]

In [9]:
home_orn = env.get_robot_ee_home_orn()
# Keeping the z a bit high to avoid collision:
z = env.table_height + CFG.blocks_block_size/2 + 0.1
#logging.critical(f"Value of z: {z}.")
orn = (0, 0.7071, 0, 0.7071)

target_ee_pose = Pose(position=(1.5 , 0.75, z), 
                         orientation=home_orn)

coordinated_path = run_coordinated_motion_planning(
    robot,
    target_ee_pose=target_ee_pose,
    collision_bodies=static_collision_bodies[1:],
    seed=CFG.seed,
    physics_client_id=physics_client_id,
    try_arm_only_first=True # Planner will try arm-only, fail, then try base+arm
)

# base_path_waypoints, arm_path_waypoints = coordinated_path
base_path_waypoints = coordinated_path


2025-08-24 19:45:06 root [WARNING] Max time reached. No IKFast solution found.
2025-08-24 19:45:06 root [WARNING] No IK solutions found in 0.501 seconds


Coordinated Planning: Arm-only IK failed. Moving to base planning.


2025-08-24 19:45:07 root [WARNING] Max time reached. No IKFast solution found.
2025-08-24 19:45:07 root [WARNING] No IK solutions found in 0.501 seconds
2025-08-24 19:45:07 predicators.pybullet_helpers.motion_planning [WARNING] 
 Collsion check:COLLIDES :-(


In [10]:
base_path_waypoints

[(0.39999716871626545, 1.5999910974242648, -1.5707959804044407),
 (0.4002081090953471, 1.609999611804352, -1.5964224030203022),
 (0.4004190494744288, 1.6200081261844395, -1.6220488256361638),
 (0.40062998985351045, 1.6300166405645267, -1.6476752482520254),
 (0.4008409302325921, 1.640025154944614, -1.6733016708678867),
 (0.40105187061167374, 1.6500336693247013, -1.6989280934837483),
 (0.40126281099075545, 1.6600421837047885, -1.7245545160996099),
 (0.4014737513698371, 1.670050698084876, -1.7501809387154714),
 (0.40168469174891874, 1.6800592124649631, -1.775807361331333),
 (0.4018956321280004, 1.6900677268450506, -1.8014337839471946),
 (0.4021065725070821, 1.7000762412251378, -1.827060206563056),
 (0.40231751288616374, 1.7100847556052252, -1.8526866291789177),
 (0.4025284532652454, 1.7200932699853124, -1.878313051794779),
 (0.40273939364432704, 1.7301017843653996, -1.9039394744106406),
 (0.4107670790310543, 1.7361947186612774, -1.9342683641518594),
 (0.41879476441778163, 1.74228765295715

In [11]:
# def preprocess_path_for_differential_drive(  
#     raw_path: List[Tuple[float, float, float]],  
#     position_threshold: float = 0.05,  
#     rotation_threshold: float = 0.1  
# ) -> List[Tuple[float, float, float]]:  
#     """  
#     Preprocess BiRRT path to separate rotation and translation phases.  
#     This mimics iGibson's three-phase motion planning approach.  
#     """  
#     if len(raw_path) < 2:  
#         return raw_path  
      
#     processed_path = [raw_path[0]]  # Start with initial pose  
      
#     for i in range(1, len(raw_path)):  
#         prev_pose = processed_path[-1]  
#         curr_pose = raw_path[i]  
          
#         x1, y1, theta1 = prev_pose  
#         x2, y2, theta2 = curr_pose  
          
#         # Calculate position and orientation differences  
#         dx, dy = x2 - x1, y2 - y1  
#         position_dist = np.sqrt(dx*dx + dy*dy)  
          
#         # Calculate target heading for translation  
#         target_heading = np.arctan2(dy, dx) if position_dist > 0.01 else theta1  
          
#         # Circular difference for angles  
#         def circular_diff(a1, a2):  
#             diff = a1 - a2  
#             while diff > np.pi: diff -= 2 * np.pi  
#             while diff < -np.pi: diff += 2 * np.pi  
#             return diff  
          
#         heading_diff = abs(circular_diff(target_heading, theta1))  
#         final_diff = abs(circular_diff(theta2, target_heading))  
          
#         # Phase 1: Rotate to face target direction (if needed)  
#         if heading_diff > rotation_threshold and position_dist > position_threshold:  
#             processed_path.append((x1, y1, target_heading))  
          
#         # Phase 2: Translate while maintaining heading (if needed)  
#         if position_dist > position_threshold:  
#             processed_path.append((x2, y2, target_heading))  
          
#         # Phase 3: Rotate to final orientation (if needed)  
#         if final_diff > rotation_threshold:  
#             processed_path.append((x2, y2, theta2))  
#         elif position_dist <= position_threshold:  
#             # Just orientation change  
#             processed_path.append((x2, y2, theta2))  
      
#     return processed_path 

# base_path_waypoints = preprocess_path_for_differential_drive(raw_path=base_path_waypoints)

In [12]:
# def deduplicate_waypoints(waypoints, tol=1e-6):
#     """
#     Remove duplicate waypoints (anywhere in list, not just consecutive) 
#     while preserving original order.
#     """
#     unique = []
#     seen = set()
#     for w in waypoints:
#         # Round to tolerance to avoid floating-point issues
#         key = tuple(round(v / tol) for v in w)
#         if key not in seen:
#             seen.add(key)
#             unique.append(w)
#     return unique

# cleaned_base_path = deduplicate_waypoints(base_path_waypoints)

In [13]:
def get_current_base_and_arm_pose(robot, state:State, objects: Sequence[Object], params: Array):

    current_base_pose = robot.get_base_pose(physics_client_id)
    current_joint_positions = robot.get_joints()

    return current_base_pose, current_joint_positions

target_base_pose = base_path_waypoints[-1]

move_option_memory = {}
params_space = Box(low=np.array([], dtype=np.float32),
                  high=np.array([], dtype=np.float32), dtype=np.float32)

move_option = create_move_base_option(robot, name="diff-drive",types=[env._robot_type], params_space=params_space,
                                    get_current_base_and_arm_pose=get_current_base_and_arm_pose, base_path=base_path_waypoints,
                                    target_base_pose = target_base_pose)


#The empty nd array is the empty param space that this option takes in as all
#the values are passed in to the option creation function. If this were to change,
#values will be passed in this array, and assigned to appropricate vars in _initialble
#and the param_space Box size will be changed accordingly.
grounded_move = move_option.ground([robot_obj], np.array([], dtype=np.float32))

In [14]:
# def _create_move_robot_base_option(name: str, robot: MobileSingleArmPyBulletRobot, 
#                                     option_types: Sequence[Type],  params_space:Box, 
#                                     env: PyBulletBlocksEnv, physics_client_id: int) -> ParameterizedOption:

#         """Compute/derive values required to initialize the base motion option which first plans
#         then executes the base motion via differential drive.
#         """

#         def get_current_base_and_arm_pose(robot: MobileSingleArmPyBulletRobot, state:State, objects: Sequence[Object],
#                                          params: Array) -> Tuple[Tuple[float, float, float], JointPositions]:

#             current_base_pose = robot.get_base_pose(physics_client_id)
#             current_joint_positions = robot.get_joints()

#             return current_base_pose, current_joint_positions

#         assert physics_client_id is not None
#         all_bodies = [p.getBodyUniqueId(i, physicsClientId=physics_client_id)
#                 for i in range(p.getNumBodies(physicsClientId=physics_client_id))]

#         # ipdb.set_trace()

#         collision_bodies = [b for b in all_bodies if b!=robot.robot_id]

#         held_obj_id_at_start = env._held_obj_id
#         ee_link_to_held_obj = None
#         if held_obj_id_at_start is not None:
#             # 1. world -> base_link pose | It actually is World -> EE transform
#             world_to_ee_pos, world_to_ee_orn = get_link_pose(
#                                                         robot.robot_id,
#                                                         robot.end_effector_id,
#                                                         physics_client_id=physics_client_id
#                                                     )

#             # 2. base_link -> world
#             ee_to_world_pos, ee_to_world_orn = p.invertTransform(
#                                                         world_to_ee_pos, world_to_ee_orn
#                                                     )
                                                    
#             # 3. world -> object
#             world_to_obj_pos, world_to_obj_orn = p.getBasePositionAndOrientation(
#                                                         held_obj_id_at_start, physicsClientId=physics_client_id
#                                                     )

#             # 4. base_link -> object (chain transforms)
#             ee_link_to_held_obj = p.multiplyTransforms(
#                                                         ee_to_world_pos, ee_to_world_orn,
#                                                         world_to_obj_pos, world_to_obj_orn
#                                                         )
#         home_orn = PyBulletBlocksEnv.get_robot_ee_home_orn()

#         return create_move_base_option(name=name, robot=robot, types=option_types, params_space=params_space, 
#             get_current_base_and_arm_pose=get_current_base_and_arm_pose, home_orn=home_orn, collision_bodies=collision_bodies, 
#             seed=CFG.seed, physics_client_id=physics_client_id, held_object_id_at_start=held_obj_id_at_start, 
#             ee_to_held_object_transform_at_start=ee_link_to_held_obj)


In [15]:
# move_option = _create_move_robot_base_option(name="MoveBase", robot=robot, option_types=[env._block_type,env._robot_type], params_space=params_space,
#                                       env=env, physics_client_id=physics_client_id)

In [16]:
# grounded_move = move_option.ground([block_obj, robot_obj], np.array([], dtype=np.float32))

In [17]:
assert grounded_move.initiable(move_option_memory)

print(grounded_move)

state = initial_state

# move_action_list = []

while not grounded_move.terminal(move_option_memory):
    action = grounded_move.policy(move_option_memory)
    # move_action_list.append(move_action_list)
    logging.warning(f"\nNext action to be simulated:{action}.")
    state = env.simulate(state, action)
    
    # omega_r, omega_l = action.base_motion['params']

    # robot.set_wheel_motors(robot, omega_r, omega_l, physics_client_id)

    # fixed_arm_pos = action.arr
    # robot.set_motors(fixed_arm_pos)

    # for _ in range(30):
    #     p.stepSimulation(physicsClientId=physics_client_id)
    # time.sleep(0.15)

'''
Now chain arm motion planning with this:
    - convert run_motion_planning to a singleParameterizedOption
    - connect to diff drive
'''

# for waypoint in arm_path_waypoints:
#     action = Action(np.zeros(len(robot.action_space.low), dtype=float))
#     action._arr = waypoint

#     robot.set_motors(action.arr)
#     robot.set_wheel_motors(robot, 0.0, 0.0, physics_client_id)

#     for _ in range(150):
#         p.stepSimulation(physicsClientId=physics_client_id)
#     time.sleep(0.05)



print(f"\n Robot at {robot.get_base_pose(physics_client_id)} after executing differential drive.")

2025-08-24 19:45:07 root [WARNING] 
Next action to be simulated:Action(_arr=array([-0.5762536 ,  0.19532396, -1.2928689 , -1.5825186 , -1.7704816 ,
       -1.8456511 ,  0.9511534 ,  0.03999988,  0.04000013], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.6817587533857854, 0.08720713204258124)}}).
2025-08-24 19:45:07 root [WARNING] 
Next action to be simulated:Action(_arr=array([-0.57625514,  0.19532414, -1.2928703 , -1.582519  , -1.7704815 ,
       -1.845651  ,  0.9511531 ,  0.04000001,  0.03999998], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.6773339309685704, 0.08901220443227977)}}).


_Option(name='diff-drive', objects=[robby:robot], params=array([], dtype=float32))
[STEP 1] NAV - d=1.984 ptr=0 look=25 αW=2.60 v=0.02 ω=0.10
[STEP 2] NAV - d=1.984 ptr=0 look=25 αW=2.60 v=0.02 ω=0.10


2025-08-24 19:45:07 root [WARNING] 
Next action to be simulated:Action(_arr=array([-0.5762552 ,  0.19532436, -1.2928717 , -1.582519  , -1.7704815 ,
       -1.845651  ,  0.95115304,  0.03999997,  0.04000002], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.6728342426935934, 0.09084670176030818)}}).
2025-08-24 19:45:07 root [WARNING] 
Next action to be simulated:Action(_arr=array([-0.57632023,  0.19530392, -1.2928977 , -1.5824598 , -1.7704515 ,
       -1.8456461 ,  0.95116955,  0.03996083,  0.04004012], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.6694081330894004, 0.09224272735170996)}}).


[STEP 3] NAV - d=1.983 ptr=0 look=25 αW=2.61 v=0.02 ω=0.10
[STEP 4] NAV - d=1.982 ptr=0 look=25 αW=2.61 v=0.02 ω=0.10


2025-08-24 19:45:07 root [WARNING] 
Next action to be simulated:Action(_arr=array([-0.5764152 ,  0.19526453, -1.2929441 , -1.5823547 , -1.7704021 ,
       -1.845639  ,  0.9512016 ,  0.03989167,  0.04011097], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.6707061802003236, 0.09171389708481711)}}).
2025-08-24 19:45:08 root [WARNING] 
Next action to be simulated:Action(_arr=array([-0.5765839 ,  0.19520675, -1.2930131 , -1.5821903 , -1.7703218 ,
       -1.845626  ,  0.95124817,  0.03978465,  0.0402206 ], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (0.6704922602325738, 0.0918010558653688)}}).


...

[STEP 403] ORIENT ONLY - d=0.032, yaw=-0.108, ω=-0.217
[STEP 404] ORIENT ONLY - d=0.032, yaw=-0.094, ω=-0.187


2025-08-24 19:46:07 root [WARNING] 
Next action to be simulated:Action(_arr=array([-0.59025675,  0.18829592, -1.300433  , -1.5635346 , -1.7635896 ,
       -1.8461177 ,  0.95583826,  0.02921116,  0.04432061], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (-0.46628009347564287, 0.46628009347564287)}}).
2025-08-24 19:46:07 root [WARNING] 
Next action to be simulated:Action(_arr=array([-0.5901559 ,  0.18833949, -1.3003734 , -1.5636449 , -1.7636424 ,
       -1.8461256 ,  0.9558078 ,  0.02928434,  0.04424567], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (-0.4030898861640087, 0.4030898861640087)}}).


[STEP 405] ORIENT ONLY - d=0.031, yaw=-0.081, ω=-0.162
[STEP 406] ORIENT ONLY - d=0.031, yaw=-0.070, ω=-0.140


2025-08-24 19:46:07 root [WARNING] 
Next action to be simulated:Action(_arr=array([-0.5900446 ,  0.18838942, -1.3003122 , -1.5637715 , -1.7637012 ,
       -1.8461347 ,  0.9557746 ,  0.02936681,  0.0441612 ], dtype=float32), extra_info={'base_motion': {'mode': 'velocity', 'params': (-0.3520480862354552, 0.3520480862354552)}}).


[STEP 407] ORIENT ONLY - d=0.031, yaw=-0.061, ω=-0.122
[TERMINAL] SUCCESS - Position error: 0.0313, Orientation error: 0.0538

 Robot at (2.1954332351193404, 0.8173803985496808, -3.030607805907387) after executing differential drive.
